In [1]:
import sys
from pathlib import Path

# resolve imports from the current working dir.
cwd = Path.cwd().resolve()
project_root = cwd.parent
sys.path.insert(0, str(project_root))

from FEX.utils.numerical_deriv import NumericalDeriv
from FEX.training.train_controller import ControllerConfig, train_network_controller
from FEX.training.train_configs import FEXConfig
from FEX.training.train_configs import runtimeconfig
from FEX.utils.tree_configs import get_tree_config

import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from FEX.utils.tree_configs import *

# Initialize logger before training
from pathlib import Path
log_path = str(Path("logs/run1/controller_eval.log"))
runtimeconfig.CreateLogger(log_path, name="train_logger")

forcing_tree_config = get_tree_config("depth_3_leaves_4_config")
inter_tree_config = get_tree_config("depth_2_tree_config")


Using device: cuda


In [2]:
import pandas as pd
import numpy as np

# Update file paths to match train_fex.py
adj_matrix = pd.read_csv('./data/BA_Nnodes100_Adj_deg_7_1.csv', header=None)
num_graph_nodes = adj_matrix.shape[0]

x_df = pd.read_csv('./data/HR_timeseries_BA_deg_7_1_SNR_45.csv', header=None)
num_timesteps, num_cols = x_df.shape
x_np = x_df.to_numpy(dtype=np.float32)
x_data = torch.from_numpy(x_np.reshape(num_timesteps, num_graph_nodes, 3))

dt = 0.01
len_run = 500
per_run_timesteps = int(len_run / dt)
cut_timestep = int(per_run_timesteps * 0.8)

num_runs = num_timesteps // per_run_timesteps
x_chunks = torch.chunk(x_data, num_runs, dim=0)
all_dx_dt = []
all_x = []

for x_run in x_chunks:
    x_run = x_run[:cut_timestep]
    dx_dt = NumericalDeriv(x_run, dt=dt) # 4th order
    x_run = x_run[2:-2]
    all_dx_dt.append(dx_dt)
    all_x.append(x_run)
all_x = all_x[:1] # take the first 1 run only
all_dx_dt = all_dx_dt[:1] # take the first 1 run only
dx_dt = torch.cat(all_dx_dt, dim=0)
x_data = torch.cat(all_x, dim=0)


In [3]:
train_x_data = x_data[:, :, :]
train_dx_dt = dx_dt[:, :, :]
adj_matrix_tensor = torch.tensor(adj_matrix.values, dtype=torch.float32).to(runtimeconfig.device)
x_data_tensor_ds = TensorDataset(train_x_data, train_dx_dt)
if runtimeconfig.device == "cuda":
    dataloader = DataLoader(x_data_tensor_ds, batch_size=32, shuffle=True, pin_memory=True)
else:
    dataloader = DataLoader(x_data_tensor_ds, batch_size=32, shuffle=True)

In [4]:
controller_config = ControllerConfig(
    input_dim=20,
    hidden_dim=64,
    lr=0.001,
    num_epochs=200,
    num_cands_per_epoch=10,
    percentile_threshold=0.4,
    num_trees=2
)
fex_config = FEXConfig(
    num_epochs=30,
    bfgs_epochs=10,
    bfgs_lr=0.1,
    leaf_dim=x_data.shape[2],
    num_leaves=forcing_tree_config.num_leaves,
    weight_decay=0.0,
    mag_entropy_weight=0.6
)
best_candidates = train_network_controller(
    forcing_tree_config,
    inter_tree_config,
    dataloader,
    adj_matrix_tensor,
    controller_config,
    fex_config,
)

RuntimeError: Cowardly refusing to serialize non-leaf tensor which requires_grad, since autograd does not support crossing process boundaries.  If you just want to transfer the data, call detach() on the tensor before serializing (e.g., putting it on the queue).

In [ ]:
import math
import os
from FEX.training.train_configs import FEXConfig
from FEX.training.train_fex import train_network_fex
from pathlib import Path
from FEX.utils.pools import GraphPool, GraphPoolCandidate
from FEX.models.nodes import UnaryOperation

saved_candidates_dir = "logs/run1/pre_finetune/best_candidates"
if os.path.isdir(saved_candidates_dir) and any(f.endswith('.pt') for f in os.listdir(saved_candidates_dir)):
    print(f"Loading saved candidates from {saved_candidates_dir} for finetuning.")
    finetune_cands = GraphPool.load_candidates(
        saved_candidates_dir,
        forcing_tree_config,
        inter_tree_config,
        device=runtimeconfig.device
    )
    print(f"Loaded {len(finetune_cands.pool)} candidates for finetuning.")

fex_config = FEXConfig(
    num_epochs = 2000,
    bfgs_epochs = 0,
    lr = 0.02,
    inter_lr = 0.008,
    bfgs_lr = 0.1,
    num_groups = 1,
    leaf_dim = x_data.shape[2],
    num_leaves = forcing_tree_config.num_leaves,
    tau_start = 4.0,
    tau_end = 0.05,
    pct_cosine_restart = 0.5,

    mag_entropy_weight = 1e-4,
    leaf_entropy_weight = 0.1,
    set_hard_at = 0.8,
    # decay_entropy_until=0.65
)
fex_config.tau_anneal_epochs = fex_config.num_epochs

"""for i, candidate in enumerate(finetune_cands):
    if i >= 1:
        # reset cand params
        print(f"candidate tree before reset: {casndidate.forcing_tree}")
        print(f"candidate tree before reset: {candidate.inter_tree}")
        candidate.forcing_tree.reset()
        candidate.inter_tree.reset()
        print(f"candidate tree after reset: {candidate.forcing_tree}")
        print(f"candidate tree after reset: {candidate.inter_tree}")

        print(f"Finetuning candidate {i} (id={candidate.id}) ...")
        score = train_network_fex(candidate.forcing_tree, candidate.inter_tree, dataloader, adj_matrix_tensor, fex_config)
        score = score
        reward = 1.0 / math.sqrt(1.0 + score)
        candidate.reward = reward
        print(f"Candidate {i} (id={candidate.id}) finetuned. Score: {score}, Reward: {reward}")

finetune_cands.sort() # resort after finetuning """

from FEX.utils.operations import UNARY_OPS
forcing_cand = None
inter_cand = None
for cand in finetune_cands:
    # print trees for cand
    if cand.id == 1:
        finetune_cand = cand
        finetune_cand.inter_tree.parent_node.right.operation = UnaryOperation(UNARY_OPS[0]).to(runtimeconfig.device)



if finetune_cand is not None:
    print(f"Testing forcing_tree id=1 and inter_tree id=4")
    finetune_cand.forcing_tree.reset()
    finetune_cand.inter_tree.reset()
    print(f"candidate forcing_tree before finetune: {finetune_cand.forcing_tree}")
    print(f"candidate inter_tree before finetune: {finetune_cand.inter_tree}")

    score = train_network_fex(finetune_cand.forcing_tree, finetune_cand.inter_tree, dataloader, adj_matrix_tensor, fex_config)
    reward = 1.0 / math.sqrt(1.0 + score)
    print(f"Finetuned combination: Score={score}, Reward={reward}")
else:
    print("Could not find candidates with id=1 and id=4")
finetune_candidate = GraphPoolCandidate(
    id=100,
    forcing_tree=finetune_cand.forcing_tree,
    inter_tree=finetune_cand.inter_tree,
    reward=reward,
)
finetune_candidates = GraphPool(pool_size=1)
finetune_candidates.add_new(finetune_candidate)
finetune_candidates.save_candidates("logs/run1/finetuned_candidates")
finetune_candidates.visualize_candidates("logs/run1/finetuned_candidates/visualizations")

Loading saved candidates from logs/run1/pre_finetune/best_candidates for finetuning.
Loaded 8 candidates for finetuning.
Testing forcing_tree id=1 and inter_tree id=4
candidate forcing_tree before finetune: (((-1.133 * identity((x[0] (conf: 0.38))) + 2.038) sub (1.018 * identity((x[1] (conf: 0.36))) + -0.867)) sub ((-2.095 * cube((x[2] (conf: 0.37))) + 0.737) add (-0.262 * square((x[1] (conf: 0.35))) + -5.225)))
candidate inter_tree before finetune: ((-1.607 * sigmoid((x[1] (conf: 0.19))) + -0.338) mul (2.332 * identity((x[0] (conf: 0.20))) + 0.494))


2026-04-18 17:11:42,101 - INFO - Adam Epoch 1/2000, Loss: 10352.1460
2026-04-18 17:11:43,055 - INFO - Adam Epoch 2/2000, Loss: 2874.2067
2026-04-18 17:11:43,991 - INFO - Adam Epoch 3/2000, Loss: 5093.6614
2026-04-18 17:11:44,896 - INFO - Adam Epoch 4/2000, Loss: 579.0513
2026-04-18 17:11:45,950 - INFO - Adam Epoch 5/2000, Loss: 685.3531
2026-04-18 17:11:46,916 - INFO - Adam Epoch 6/2000, Loss: 167.8735
2026-04-18 17:11:47,859 - INFO - Adam Epoch 7/2000, Loss: 1558.1559
2026-04-18 17:11:48,818 - INFO - Adam Epoch 8/2000, Loss: 103.6731
2026-04-18 17:11:49,722 - INFO - Adam Epoch 9/2000, Loss: 107.8283
2026-04-18 17:11:50,653 - INFO - Adam Epoch 10/2000, Loss: 48.4359
2026-04-18 17:11:51,682 - INFO - Adam Epoch 11/2000, Loss: 132.7236
2026-04-18 17:11:52,564 - INFO - Adam Epoch 12/2000, Loss: 39.5869
2026-04-18 17:11:53,441 - INFO - Adam Epoch 13/2000, Loss: 193.3326
2026-04-18 17:11:54,322 - INFO - Adam Epoch 14/2000, Loss: 44.8123
2026-04-18 17:11:55,222 - INFO - Adam Epoch 15/2000, Lo

Finetuned combination: Score=3.2393139857280104, Reward=0.4856822234445528


AttributeError: 'bool' object has no attribute 'lower'

In [ ]:
"""finetune_cands.visualize_candidates(directory="logs/run1/finetuned/candidates_viz")
finetune_cands.save_candidates("logs/run1/finetuned/best_candidates")"""

AttributeError: 'FEX' object has no attribute 'visualize_tree'